In [8]:
import numpy as np
import random

In [9]:
import numpy as np

# Environment: Cloud Data Center
# -------------------------------
class CloudEnvironment:
    def __init__(self, num_servers=3, server_capacity=100, power_base=50, power_per_cpu=1):
        self.num_servers = num_servers
        self.server_capacity = server_capacity
        self.power_base = power_base
        self.power_per_cpu = power_per_cpu
        self.reset()

    def reset(self):
        self.servers = np.zeros(self.num_servers)  # CPU usage of each server
        self.total_power = 0
        return self._get_state()

    def _get_state(self):
        return tuple(self.servers)

    def step(self, vm_demand, action):
        # Try placing VM on selected server
        reward = 0
        done = False

        if self.servers[action] + vm_demand <= self.server_capacity:
            self.servers[action] += vm_demand
            power = self.power_base + self.power_per_cpu * self.servers[action]
            reward = -power  # less power = higher reward
        else:
            reward = -200  # penalty for invalid placement

        self.total_power += -reward
        return self._get_state(), reward, done


In [10]:
import random
import numpy as np

# Q-learning Agent
# -------------------------------
class QLearningAgent:
    def __init__(self, env, alpha=0.1, gamma=0.9, epsilon=0.2):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = {}

    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.env.num_servers - 1)
        qs = [self.get_q(state, a) for a in range(self.env.num_servers)]
        return int(np.argmax(qs))

    def learn(self, state, action, reward, next_state):
        old_q = self.get_q(state, action)
        next_max_q = max([self.get_q(next_state, a) for a in range(self.env.num_servers)])
        new_q = old_q + self.alpha * (reward + self.gamma * next_max_q - old_q)
        self.q_table[(state, action)] = new_q


In [11]:
# Training loop
# -------------------------------
env = CloudEnvironment()
agent = QLearningAgent(env)

import random
import numpy as np

# Training Loop
# -------------------------------
for episode in range(1000):
    state = env.reset()
    total_reward = 0

    for step in range(5):  # 5 VMs per episode
        vm_demand = random.randint(10, 40)
        action = agent.choose_action(state)
        next_state, reward, done = env.step(vm_demand, action)
        agent.learn(state, action, reward, next_state)
        state = next_state
        total_reward += reward

    if episode % 100 == 0:
        print(f"Episode {episode}: Total Reward = {total_reward:.2f}, Power Used = {env.total_power:.2f}")

print("\nTraining completed! Agent learned energy-efficient VM placement.")


Episode 0: Total Reward = -582.00, Power Used = 582.00
Episode 100: Total Reward = -466.00, Power Used = 466.00
Episode 200: Total Reward = -414.00, Power Used = 414.00
Episode 300: Total Reward = -483.00, Power Used = 483.00
Episode 400: Total Reward = -486.00, Power Used = 486.00
Episode 500: Total Reward = -519.00, Power Used = 519.00
Episode 600: Total Reward = -561.00, Power Used = 561.00
Episode 700: Total Reward = -430.00, Power Used = 430.00
Episode 800: Total Reward = -455.00, Power Used = 455.00
Episode 900: Total Reward = -421.00, Power Used = 421.00

Training completed! Agent learned energy-efficient VM placement.
